<a href="https://colab.research.google.com/github/warry258/colab/blob/main/%E6%89%B9%E9%87%8F%E6%8F%90%E5%8F%96%E6%8E%A7%E5%88%B6%E5%9B%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive 挂载完成！")

In [ ]:
# @title 2. 环境配置与下载预处理模型
import os
import sys

# 1. 安装核心依赖（仅需轻量级底层库，抛弃臃肿的框架）
!pip install -q controlnet_aux==0.0.9 opencv-python-headless pillow accelerate

import torch
import cv2
import numpy as np
from PIL import Image
from controlnet_aux import (
    CannyDetector,
    OpenposeDetector,
    HEDdetector,
    MLSDdetector,
    MidasDetector,
    PidiNetDetector
)

# 2. 环境检测
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✅ 当前运算环境: 【{device.upper()}】")

# 设置模型缓存目录
os.environ["HF_HOME"] = "/content/ckpts"
os.makedirs("/content/ckpts", exist_ok=True)
os.makedirs("/content/input_images", exist_ok=True)
os.makedirs("/content/output_controls", exist_ok=True)

# 3. 预加载所有核心模型字典
preprocessors = {}

def load_models():
    print("⏳ 正在拉取并加载预处理器模型 (初次执行需下载少量文件，请稍候)...")
    try:
        preprocessors["canny"]   = CannyDetector()
        preprocessors["pose"]    = OpenposeDetector.from_pretrained("lllyasviel/Annotators").to(device)
        preprocessors["depth"]   = MidasDetector.from_pretrained("lllyasviel/Annotators").to(device)
        preprocessors["mlsd"]    = MLSDdetector.from_pretrained("lllyasviel/Annotators").to(device)
        preprocessors["hed"]     = HEDdetector.from_pretrained("lllyasviel/Annotators").to(device)
        preprocessors["pidinet"] = PidiNetDetector.from_pretrained("lllyasviel/Annotators").to(device)
        print("🎉 所有模型加载就绪！")
    except Exception as e:
        print(f"❌ 模型加载发生异常: {e}")

load_models()

In [ ]:
# @title 3. 图像上传与批量提取控制图 (交互面板)
import io
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# ================= 界面组件 =================
w_source_type = widgets.RadioButtons(options=['从下方按钮上传图片', '读取服务器/云盘目录路径'], value='从下方按钮上传图片', description='输入方式:', style={'description_width': '80px'})
w_upload = widgets.FileUpload(accept='image/*', multiple=True, button_style='primary', description='点击上传图片')
w_path = widgets.Text(placeholder='/content/drive/MyDrive/MyImages', description='目录路径:', style={'description_width': '80px'}, layout=widgets.Layout(width='500px'))

w_resolution = widgets.IntSlider(value=1024, min=512, max=2048, step=64, description="处理分辨率", style={'description_width': '80px'})

# 提取器开关及参数
w_do_canny = widgets.Checkbox(value=False, description="Canny (线稿)", layout=widgets.Layout(width='150px'))
w_canny_low = widgets.IntSlider(value=100, min=0, max=255, description="低阈值")
w_canny_high = widgets.IntSlider(value=200, min=0, max=255, description="高阈值")

w_do_depth = widgets.Checkbox(value=False, description="Depth (深度)", layout=widgets.Layout(width='150px'))

w_do_pose = widgets.Checkbox(value=False, description="Pose (姿态)", layout=widgets.Layout(width='150px'))
w_pose_hand = widgets.Checkbox(value=False, description="提取手部")
w_pose_face = widgets.Checkbox(value=False, description="提取面部")

w_do_mlsd = widgets.Checkbox(value=False, description="MLSD (直线)", layout=widgets.Layout(width='150px'))
w_mlsd_v = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, description="得分阈值")
w_mlsd_d = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, description="距离阈值")

w_do_hed = widgets.Checkbox(value=False, description="HED (软边缘)", layout=widgets.Layout(width='150px'))

w_do_scribble = widgets.Checkbox(value=False, description="Scribble (涂鸦)", layout=widgets.Layout(width='150px'))
w_scribble_mode = widgets.Dropdown(options=["HED", "PidiNet"], value="HED", description="底层算法")

btn_run = widgets.Button(description="🚀 开始批量提取", button_style="success", layout=widgets.Layout(width='200px', height='40px'))
out_log = widgets.Output()

# 动态显示逻辑
def on_source_change(c):
    if '上传' in c['new']:
        w_upload.layout.display = 'block'
        w_path.layout.display = 'none'
    else:
        w_upload.layout.display = 'none'
        w_path.layout.display = 'block'
w_source_type.observe(on_source_change, names='value')
on_source_change({'new': w_source_type.value})

# ================= 核心处理逻辑 =================
def process_images(_):
    with out_log:
        clear_output()
        image_paths =[]

        # 1. 收集目标图片
        if '上传' in w_source_type.value:
            if not w_upload.value:
                print("⚠️ 请先上传图片！")
                return
            # 保存上传的图片到本地缓存
            for name, file_info in w_upload.value.items():
                content = file_info['content'] if isinstance(w_upload.value, dict) else file_info
                temp_path = os.path.join("/content/input_images", name)
                with open(temp_path, "wb") as f:
                    f.write(content)
                image_paths.append(temp_path)
            print(f"📥 成功接收 {len(image_paths)} 张上传图片。")
        else:
            dir_path = w_path.value.strip()
            if not os.path.isdir(dir_path):
                print(f"⚠️ 找不到目录: {dir_path}")
                return
            valid_exts = ('*.jpg', '*.jpeg', '*.png', '*.webp')
            for ext in valid_exts:
                image_paths.extend(glob.glob(os.path.join(dir_path, ext)))
                image_paths.extend(glob.glob(os.path.join(dir_path, ext.upper())))
            print(f"📂 在目录中找到 {len(image_paths)} 张图片。")

        if not image_paths:
            print("⚠️ 未找到有效图片文件。")
            return

        res = w_resolution.value
        out_dir = "/content/output_controls"

        # 2. 遍历执行提取
        for idx, img_path in enumerate(image_paths):
            filename = os.path.basename(img_path)
            name_no_ext = os.path.splitext(filename)[0]
            print(f"\n[{idx+1}/{len(image_paths)}] 正在处理: {filename} ...")

            try:
                img = Image.open(img_path).convert("RGB")
            except Exception as e:
                print(f"  ❌ 读取失败: {e}")
                continue

            # --- Canny ---
            if w_do_canny.value:
                print("  -> 提取 Canny...")
                out = preprocessors["canny"](img, low_threshold=w_canny_low.value, high_threshold=w_canny_high.value, image_resolution=res)
                out.save(os.path.join(out_dir, f"{name_no_ext}_canny.png"))

            # --- Depth ---
            if w_do_depth.value:
                print("  -> 提取 Depth...")
                out = preprocessors["depth"](img, image_resolution=res)
                out.save(os.path.join(out_dir, f"{name_no_ext}_depth.png"))

            # --- Pose ---
            if w_do_pose.value:
                print("  -> 提取 Pose...")
                out = preprocessors["pose"](img, include_body=True, include_hand=w_pose_hand.value, include_face=w_pose_face.value, image_resolution=res)
                out.save(os.path.join(out_dir, f"{name_no_ext}_pose.png"))

            # --- MLSD ---
            if w_do_mlsd.value:
                print("  -> 提取 MLSD...")
                out = preprocessors["mlsd"](img, thr_v=w_mlsd_v.value, thr_d=w_mlsd_d.value, image_resolution=res)
                out.save(os.path.join(out_dir, f"{name_no_ext}_mlsd.png"))

            # --- HED ---
            if w_do_hed.value:
                print("  -> 提取 HED...")
                out = preprocessors["hed"](img, image_resolution=res)
                out.save(os.path.join(out_dir, f"{name_no_ext}_hed.png"))

            # --- Scribble ---
            if w_do_scribble.value:
                print(f"  -> 提取 Scribble ({w_scribble_mode.value})...")
                algo = preprocessors["hed"] if w_scribble_mode.value == "HED" else preprocessors["pidinet"]
                out = algo(img, scribble=True, image_resolution=res)
                out.save(os.path.join(out_dir, f"{name_no_ext}_scribble.png"))

        print("\n✅ 所有图片批量处理完毕！请运行下方导出单元格。")

btn_run.on_click(process_images)

# ================= UI 布局渲染 =================
ui = widgets.VBox([
    widgets.HTML("<h3>🎨 ControlNet 批量提取面板</h3><hr>"),
    w_source_type, w_upload, w_path,
    widgets.HTML("<hr><b>🖼️ 全局分辨率:</b>"), w_resolution,
    widgets.HTML("<hr><b>🎯 提取配置:</b>"),
    widgets.HBox([w_do_canny, widgets.VBox([w_canny_low, w_canny_high])]),
    widgets.HBox([w_do_depth]),
    widgets.HBox([w_do_pose, w_pose_hand, w_pose_face]),
    widgets.HBox([w_do_mlsd, widgets.VBox([w_mlsd_v, w_mlsd_d])]),
    widgets.HBox([w_do_hed]),
    widgets.HBox([w_do_scribble, w_scribble_mode]),
    widgets.HTML("<hr>"),
    btn_run, out_log
])
display(ui)

In [ ]:
# @title 4. 一键导出提取结果 (保存至网盘或下载到本地)
import os
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

w_dest_path = widgets.Text(value="/content/drive/MyDrive/ControlNet_提取结果", description="云盘目标目录", style={'description_width': '80px'}, layout=widgets.Layout(width='400px'))
btn_copy_drive = widgets.Button(description="📂 拷贝至 Google Drive", button_style="success", layout=widgets.Layout(width='180px'))
btn_download_zip = widgets.Button(description="📦 打包下载到本地", button_style="info", layout=widgets.Layout(width='180px'))
out_export = widgets.Output()

out_dir = "/content/output_controls"

def copy_to_drive(_):
    with out_export:
        clear_output()
        if not os.path.exists(out_dir) or not os.listdir(out_dir):
            print("⚠️ 暂无提取结果，请先在上方执行提取操作。")
            return
        dest = w_dest_path.value.strip()
        os.makedirs(dest, exist_ok=True)
        !cp -r {out_dir}/* "{dest}/"
        print(f"✅ 已成功拷贝 {len(os.listdir(out_dir))} 个文件至：{dest}")

def download_local(_):
    with out_export:
        clear_output()
        if not os.path.exists(out_dir) or not os.listdir(out_dir):
            print("⚠️ 暂无提取结果，请先在上方执行提取操作。")
            return
        print("⏳ 正在打包 zip 文件，请稍候...")
        shutil.make_archive("/content/Extracted_ControlNet", 'zip', out_dir)
        print("✅ 打包完成！浏览器即将开始下载...")
        files.download("/content/Extracted_ControlNet.zip")

btn_copy_drive.on_click(copy_to_drive)
btn_download_zip.on_click(download_local)

ui_export = widgets.VBox([
    widgets.HTML("<h3>💾 导出与下载</h3>"),
    widgets.HBox([w_dest_path, btn_copy_drive]),
    widgets.HTML("<br><i>或者直接下载到本地电脑：</i>"),
    btn_download_zip,
    out_export
])
display(ui_export)